# LangGraph com Memória AgentCore - Human in the Loop (Memória de curto prazo)

## Introdução
Este notebook demonstra como integrar as capacidades de Memória do Amazon Bedrock AgentCore com LangGraph para criar fluxos de trabalho **human-in-the-loop**. Vamos focar na persistência de **memória de curto prazo** combinada com a capacidade de interromper a execução do agente para intervenção humana, criando cenários sofisticados de suporte ao cliente com transições transparentes.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Curto Prazo Conversacional                                                       |
| Caso de uso do agente | Suporte ao Cliente com Escalonamento Humano                                   |
| Framework Agêntico  | Langgraph                                                                        |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | AgentCoreMemorySaver, Breakpoints para Intervenção Humana                    |
| Complexidade do exemplo | Intermediária                                                                |

Você aprenderá a:
- Combinar checkpointing de Memória AgentCore com interrupções de fluxo de trabalho
- Implementar pontos de intervenção humana com contexto de memória persistido
- Retomar fluxos de trabalho interrompidos com atualizações do supervisor humano
- Construir pipelines de suporte ao cliente com assistência humana

## Pré-requisitos

- Python 3.10+
- Conta AWS com permissões apropriadas
- Acesso aos modelos do Amazon Bedrock

Vamos começar!

In [ ]:
# Install necessary libraries
!pip install -qr requirements.txt

In [ ]:
# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# Imports that enable human-in-the-loop implementation
from langgraph.types import Command, interrupt

In [ ]:
import os
import logging

from bedrock_agentcore.memory import MemoryClient
# Import the AgentCoreMemorySaver that we will use as a checkpointer
from langgraph_checkpoint_aws import AgentCoreMemorySaver

logging.getLogger("support-agent").setLevel(logging.INFO)
region = os.getenv('AWS_REGION', 'us-west-2')

logger = logging.getLogger("support-agent")

## Passo 1: Criação de Memória
Nesta seção, vamos criar um armazenamento de memória usando o AgentCore Memory. Isso nos permitirá persistir o histórico de conversas e recuperá-lo conforme necessário.

In [ ]:
memory_name = "SupportAgent"

client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory["id"]

### Configuração de Memória AgentCore

Agora vamos configurar nosso checkpointer de Memória AgentCore que persistirá automaticamente o estado do agente entre turnos e durante interrupções.

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize checkpointer for state persistence
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

## Passo 2: Ferramenta Human-in-the-Loop
Vamos definir as ferramentas que nosso agente de suporte usará. A ferramenta chave é o `request_human_assistance` que aciona um breakpoint para revisão do supervisor.

In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b

@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply, human_assistance]

## Passo 3: Implementação do Agente LangGraph

Agora vamos criar nosso agente de suporte usando o framework ReAct do LangGraph com suporte a breakpoints.

In [ ]:
# Initialize LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## Passo 4: Executar o Agente de Suporte
Agora podemos executar o agente com nosso sistema de Memória AgentCore integrado e capacidades human-in-the-loop.

In [ ]:
user_input = "I would like to work with a customer service human agent."
config = {"configurable": {"thread_id": "1", "actor_id": "demo-notebook"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

### Interrupção do Fluxo de Trabalho

Observe como a execução pausou quando a ferramenta de assistência humana foi chamada. O estado do agente, incluindo todo o contexto da conversa, é automaticamente salvo na Memória AgentCore.

In [ ]:
snapshot = graph.get_state(config)
snapshot.next

### Intervenção do Supervisor Humano

Agora vamos agir como o supervisor humano e fornecer orientação para o agente continuar.

In [ ]:
human_response = (
    "I'm sorry to hear that you are frustrated. Looking at the past conversation history, I can see that you've requested a refund. I've gone ahead and credited it to your account."
)

human_command = Command(resume={"messages": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

## Resumo

Neste notebook, demonstramos:

1. Como criar um checkpointer de Memória AgentCore com breakpoints para interrupções de fluxo de trabalho
2. Como implementar ferramentas human-in-the-loop que pausam a execução do agente
3. Como retomar agentes interrompidos com atualizações de estado humano
4. Como a Memória AgentCore preserva todo o contexto durante interrupções do fluxo de trabalho

Esse padrão é poderoso para aplicações do mundo real onde certas ações requerem aprovação ou orientação humana.

## Limpeza
Vamos deletar a memória para limpar os recursos usados neste notebook.

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)